# 20.22. Аудит подготовки индивидуальной КТ/FEM-модели Георгия

Дата анализа: 13 октября 2026 года. Независимый анализ исходников выполняет
Astra; проверку происхождения сохранённых артефактов и оформление отчёта
выполняет основной агент. Аудит завершён; найденные проблемы и рекомендуемые
исправления изложены ниже. Общая модель при этом не изменялась.

Главный результат — выявлен недостаток контроля формы тетраэдров при
многократном уточнении. Найдены две воспроизводимые ошибки кода, проявление
которых в завершённой текущей цепочке не установлено. Кроме того, описание
постоянных физических электродов на руках не отделено от выбранного
электрического подключения. Влияние этих проблем на импеданс требует
разделяющих проверок; причина провала сеточной сходимости пока не определена.

Цель — установить, какие ошибки подготовки трёхмерной модели могут влиять
на прямой расчёт импеданса и на восстановление удельных сопротивлений и
положения боковых электродных сборок. Рассматривается вся последовательность:
выбор КТ и масок тканей, наружный контур, цилиндрические руки, конечные
электродные контакты, объёмная сетка, интегрирование долей тканей и экспорт
материальных операторов. Модель Георгия проверяется по фактически исполненным
версиям кода; подход Ника используется для сопоставления алгоритмов.

Рассматриваемый КТ-том имеет размер 512 × 512 × 280 вокселей и шаг
0,879 × 0,879 × 1,25 мм. Получение КТ на глубоком вдохе с задержкой дыхания
подтверждено автором. Маски лёгких, костей, сердца и жира получены автоматически;
их независимая анатомическая точность не установлена. Оставшаяся область
принимается мягкотканным фоном. Обхват обеих рук 370 мм на уровне токовых
электродов сообщён автором; круговые цилиндры радиусом 58,89 мм являются
геометрическим допущением. Положение общего центра боковых сборок
реконструируется, его погрешность независимо не измерена.

Координаты модели заданы в системе LPS: ось x направлена влево, y — кзади,
z — к голове; единица геометрии — миллиметр. Перед сборкой электрических
операторов геометрия переводится в метры. Удельное сопротивление обозначено
ρ и выражено в Ом·м, передаточный импеданс Z — в Ом. Объёмный элемент сетки
имеет форму тетраэдра; его тканевые доли задают усреднённую проводимость
в пределах элемента. Доказанная положительность объёма элемента, точность
этого усреднения и сеточная сходимость проверяются раздельно.

Сохранённый десятичасовой подбор и его численные проверки описаны в
[отчёте 33.28](33.28_Совместный_FEM_подбор_за_10_часов_Георгия_Scorpion_v2.html).
Настоящий аудит не изменяет исторические результаты, общую модель и её
статус принятия. Новые FEM-расчёты при подготовке отчёта не выполняются.

## Происхождение общей модели и рост сетки

Совпадение контрольных сумм проверено для всех 91 файла, закреплённого
паспортом общей модели. Связь десятичасового расчёта с 16 сохранёнными
версиями исходников, 24 экспериментальными уровнями импеданса и фиксированным
кандидатом численных проверок также подтверждена. Это проверка происхождения
и согласованности артефактов. Анатомическая и физическая валидация из неё
не следует. Строгий запрос модели для количественной инверсии отклоняется
действующим контрактом из-за нерешённой точности FEM.

В таблице 2 приведены последовательные сетки с одной наружной геометрией.
Обозначение «6 мм» относится к исходной подготовке, а «3 мм» — к локальному
объёмному уточнению возле контактов. Они не означают одинаковую длину всех
рёбер. Минимальные объёмы приведены из паспортов; самостоятельный контроль
формы и обусловленности элементов является частью аудита. Проверка варианта
2 мм не была завершена в пределах десятичасового расчёта.


In [1]:
from IPython.display import display
from build_georg_model_preparation_audit import reference_table, lineage_table
display(reference_table())
display(lineage_table())

Проверка,Результат
Закреплённые файлы,91
Совпадение контрольных сумм,91
Независимая точность анатомии,Не подтверждена
Сеточная сходимость,Не подтверждена
Количественная инверсия в строгом режиме,Не разрешена текущим численным статусом
Изменение общей модели при аудите,Не выполнялось


Этап,Узлы,Тетраэдры,"Минимальный объём по паспорту, мм³",Числа точек интегрирования долей тканей
"Исходная объёмная сетка, 6 мм",246 718,1 302 197,Не указан,Не применяется
Сетка после уточнения фиксированных контактов,619 319,2 135 211,"7,70206·10⁻¹⁰",256 / 1024
Сетка всех боковых контактов,1 070 113,3 068 757,"7,70206·10⁻¹⁰",1024 / 4096
Сетка для поиска на правой боковой поверхности,2 564 219,6 720 895,"1,06347·10⁻¹⁰",1024 / 4096
"Локальное объёмное уточнение, 3 мм",6 782 671,29 897 428,"1,7943·10⁻¹³",4096 / 16384



## Результаты независимого анализа

### 1. Последовательность подготовки модели

В таблице 3 приведён фактически выполненный путь. Обозначение «подтверждено»
относится к указанной вычислительной проверке. Оно не распространяется
автоматически на точность анатомии или пригодность модели для инверсии.

<details><summary>Таблица 3. Подготовка КТ/FEM-модели и границы выполненных проверок</summary>
<div class="audit-table"><table>
<thead><tr><th>Этап</th><th>Фактически выполненная операция</th><th>Результат проверки</th></tr></thead>
<tbody>
<tr><td>Исходная КТ</td><td>Статическая серия 301, 280 срезов; преобразование плотности с наклоном 1 и смещением −1024 HU</td><td>Пространственные соглашения согласованы; происхождение серии прослежено Astra</td></tr>
<tr><td>Тканевые маски</td><td>Автоматическое выделение и объединение классов лёгких, костей, сердца и жира; TotalSegmentator 2.18.0</td><td>Размеры и аффинные преобразования всех четырёх масок совпадают с КТ; анатомическая точность не подтверждена</td></tr>
<tr><td>Наружный контур</td><td>Порог −450 HU, морфологическое открытие, выбор компоненты, заполнение полостей, упрощение и локальный ремонт</td><td>Воспроизводимость и замкнутость проверены; возможны дефекты поверхностных выступов и отказ на пустом срезе</td></tr>
<tr><td>Цилиндрические руки</td><td>Поиск заглублённого полного сечения и булево объединение с телом</td><td>Геометрическое покрытие стыков полное; положение и тканевые свойства рук приняты как допущения</td></tr>
<tr><td>Электроды</td><td>Проекция центров на поверхность, построение площадок номинального диаметра 5 мм, объединение площадок в электрические группы</td><td>Уточнённые верхние и нижние координаты согласованы; постоянный физический набор описан неполно</td></tr>
<tr><td>Объёмная сетка</td><td>TetGen 0.8.3 и последовательное согласованное деление рёбер</td><td>Граница и объём сохраняются; форма тетраэдров ухудшается и не имеет достаточного контроля</td></tr>
<tr><td>Тканевые доли</td><td>Отображение воксельных масок на тетраэдры; вложенные выборки Соболя и проверка однородности воксельных областей</td><td>В проверенных примерах ошибка сертификатора не обнаружена; пространственное разрешение материалов меняется при уточнении</td></tr>
<tr><td>FEM-операторы</td><td>Перевод в SI, пять материальных матриц, отдельная матрица полной электродной модели CEM</td><td>Единицы и индексы согласованы; повторное использование правил интегрирования имеет ошибочную ветку</td></tr>
<tr><td>Общий вход будущих расчётов</td><td>Паспорт модели с проверкой контрольных сумм и отдельным численным статусом</td><td>Отрицательный результат сходимости сохранён; строгая инверсия блокируется</td></tr>
</tbody></table></div></details>

### 2. Качество формы тетраэдров

**Подтверждённый недостаток контроля.** Проверки положительности объёма,
замкнутости и сохранения границы допускают крайне плоские элементы.
В текущем уточнении не установлен критерий, ограничивающий их геометрическую
обусловленность. Поэтому увеличение числа элементов не гарантирует улучшения
численной пригодности сетки.

Для диагностики из каждой сохранённой сетки независимо выбраны 10 000
тетраэдров генератором с начальным значением 1729. Выборка равномерна по
номерам элементов, без взвешивания объёмом. Для каждого элемента построена
матрица E из трёх векторов рёбер, выходящих из его первой вершины. Её число
обусловленности определяется отношением наибольшего сингулярного числа к
наименьшему:

$$
\kappa_2(E)=\frac{s_{\max}(E)}{s_{\min}(E)}.
$$

где sₘₐₓ и sₘᵢₙ — наибольшее и наименьшее сингулярные числа матрицы рёбер.
Этот показатель характеризует
геометрию элемента и преобразование градиентов. Он не является измеренной
обусловленностью всей FEM-системы. Astra и основной агент получили одинаковые
значения медиан и 95-х процентилей.

<details><summary>Таблица 4. Независимо воспроизведённая диагностика сохранённых сеток</summary>
<div class="audit-table"><table>
<thead><tr><th>Сетка</th><th>Медиана κ₂(E), выборка 10 000</th><th>95-й процентиль κ₂(E)</th><th>Число элементов объёмом менее 10⁻⁹ мм³, проверка всей сетки</th></tr></thead>
<tbody>
<tr><td>После уточнения фиксированных контактов, 2,14 млн элементов</td><td>4,815</td><td>1,619·10⁵</td><td>2</td></tr>
<tr><td>Поисковая, 6,72 млн элементов</td><td>69,577</td><td>3,041·10⁵</td><td>220</td></tr>
<tr><td>Локальное объёмное уточнение, 29,90 млн элементов</td><td>4382,865</td><td>1,410·10⁶</td><td>1 039 210</td></tr>
</tbody></table></div></details>

Неположительные объёмы в независимой проверке не найдены. Порог 10⁻⁹ мм³
использован как диагностический показатель роста числа малых элементов;
новым критерием принятия сетки он не назначается. Их число не определяет
суммарную объёмную долю плохих элементов. Подтверждение определителя
с повышенной точностью проверяет объём, но не улучшает форму элемента.

Эта диагностика описывает форму элементов. Она сама по себе не доказывает,
что сдвиг импедансов на 2–5 Ом обусловлен ошибкой вычисления локального
определителя или обратной матрицы. Для установления причины требуется
отдельная численная проверка.

**Возможное влияние и исправление.** Плохая форма может ухудшать численные
операторы, производные и скорость решения; величина эффекта ещё не отделена
от других причин. Нужна потоковая статистика качества по областям и материалам:
нормированная форма, углы, обусловленность и близость вершин, особенно на
стыках рук и возле боковых контактов. Следующий кандидат сетки должен
проходить количественный контроль формы. Дальнейшее деление существующих
плохих элементов нельзя считать достаточным исправлением; удаление элементов
без восстановления границы также недопустимо.

Источники: [историческая подготовка TetGen](../../MATLAB_TRKG4_real_subjects/output/exploratory/georg_ct_model_20261011/code_sources/3ad5d807849c2081b832259006eaf95a0284879599b819aac330e4451f7185bb.py),
[алгоритм деления рёбер](../../MATLAB_TRKG4_real_subjects/tools/refine_surface_contact_mesh.py),
[проверка объёмов элементов](../../MATLAB_TRKG4_real_subjects/tools/georg_ct_fem_prepare.py).

### 3. Сеточное сравнение одновременно меняет представление тканей

**Подтверждённый факт вычислительной постановки.** После деления рёбер новые
дочерние элементы повторно классифицируются по маскам. Поэтому текущая
проверка сравнивает и пространство решения для потенциала, и распределение
усреднённой проводимости. Плохая сеточная сходимость остаётся фактом, но эти
два источника расхождения пока не разделены.

Astra вычислила суммарные тканевые объёмы для одного правила интегрирования
с 16 384 точками. Массовой потери ткани между поисковой сеткой и локальным
уточнением не обнаружено.

<details><summary>Таблица 5. Объёмы тканей на двух сетках по проверке Astra</summary>
<div class="audit-table"><table>
<thead><tr><th>Материал</th><th>Поисковая сетка, мл</th><th>Локальное уточнение, мл</th><th>Изменение, мл</th></tr></thead>
<tbody>
<tr><td>Мягкотканный фон</td><td>17 645,428</td><td>17 645,956</td><td>+0,529</td></tr>
<tr><td>Лёгкие</td><td>6037,636</td><td>6037,696</td><td>+0,061</td></tr>
<tr><td>Кости</td><td>1442,370</td><td>1442,540</td><td>+0,169</td></tr>
<tr><td>Сердце</td><td>650,457</td><td>650,457</td><td>0</td></tr>
<tr><td>Жир</td><td>13 330,103</td><td>13 329,344</td><td>−0,758</td></tr>
</tbody></table></div></details>

Значения округлены независимо; изменение рассчитано до округления.
Близость общих объёмов не подтверждает одинакового локального распределения
проводимости возле электродов. В линейном тетраэдре тканевые доли задают
среднюю проводимость. Уточнение неоднородной области создаёт больше
пространственных степеней свободы, даже если суммарные объёмы сохраняются.
Успешное сравнение 4096 и 16 384 точек проверяет интегрирование долей на
фиксированной сетке, а не достаточность её пространственного разрешения.

**Разделяющая проверка.** На одной дочерней сетке сначала наследовать
проводимость каждого родительского элемента, затем заменить её долями,
повторно вычисленными по КТ. Сравнение родительской сетки с первым вариантом
выделит изменение пространства потенциала; сравнение двух дочерних вариантов
покажет эффект переназначения материалов. Эти новые FEM-расчёты при аудите
не выполнялись.

Источники: [отображение дочерних элементов на маски](../../MATLAB_TRKG4_real_subjects/tools/georg_lateral_candidate_mesh.py),
[сборка материальных операторов](../../MATLAB_TRKG4_real_subjects/src/trkg4_georg_check_materials.m).

### 4. Ошибка выбора правила интегрирования при повторном использовании матриц

**Подтверждённая ошибка кода для будущих вариантов.** При совпадении сеток
в первый ряд дочернего набора переносится второй ряд родительских матриц.
Выбор выполняется по номеру ряда, без проверки фактического числа точек
интегрирования. Для родителя 256/1024 и дочернего набора 1024/4096 это
корректно. Для родителя 1024/4096 тот же код подставляет матрицы 4096 в
дочернее правило, названное 1024. Разница между правилами может искусственно
исчезнуть.

В фактической цепочке десятичасового расчёта одинаковая сетка используется
после родителя 256/1024. Затем поисковая сетка меняется и оба ряда матриц
собираются заново. Найденная ошибка не опровергает выполненную текущую
проверку интегрирования. Однако она делает небезопасным следующий вариант
повторного использования матриц на неизменной сетке.

**Исправление и проверка.** Сопоставлять правила по явно сохранённому числу
точек и контрольной сумме соответствующей таблицы долей. Если точного
соответствия нет, нужный ряд матриц собирать заново. Небольшой тест должен
охватить обе пары родительских правил и обнаруживать неверную подстановку.

Источник: [выбор родительских материальных матриц](../../MATLAB_TRKG4_real_subjects/src/trkg4_georg_lateral_materials.m).

### 5. Ошибка обработки пустого аксиального среза

**Воспроизведённая ошибка кода.** Если после морфологического открытия на
срезе нет ненулевой связной компоненты, выбор максимальной компоненты
возвращает индекс фоновой области. Весь фон становится телом. Аналогичная
защита отсутствует и для полностью пустого трёхмерного входа.

Основной агент повторил пример Astra на массиве 32 × 32 × 3: крайние
срезы заполнены воздухом, квадрат тела размером 16 × 16 находится только
на среднем. На входе получено 0, 256 и 0 вокселей тела; после обработки —
1024, 244 и 1024. Морфология среднего среза закономерно меняет углы квадрата,
а заполнение крайних срезов является ошибкой.

Для сохранённого тела Георгия полностью заполненных срезов нет. Независимо
подтверждено, что число вокселей тела на срезе лежит между 104 749 и 136 697.
Следовательно, проявление именно этого отказа в текущем контуре не обнаружено.

**Исправление.** До выбора максимальной компоненты отдельно проверять
наличие ненулевого компонента. Пустой срез должен оставаться пустым;
полностью пустой вход должен завершаться явным отказом подготовки поверхности.
Нужны тесты пустого входа, пустых краёв и среза, исчезающего после открытия.
При другом КТ эта ошибка может создавать искусственные проводящие пластины.

Источник: [выбор связной компоненты контура](../../MATLAB_TRKG4_real_subjects/tools/georg_ct_model.py).

### 6. Физические электроды и электрические группы

**Подтверждённое несоответствие экспериментальному описанию.** Автор уточнил,
что физические электроды на руках не перемещались, менялись варианты
соединения групп. Текущая CEM, то есть полная электродная модель с конечной
площадью и контактным импедансом, получает разные наборы площадок для
разных подключений. Отдельного постоянного списка физических площадок нет.

Независимо проверены сохранённые координаты. Верхние U монтажа 1×1 точно
совпадают с соответствующими U монтажа 1×4. Верхние и нижние контакты
2×2 согласованы с уточнённой геометрией. Ошибочного повторного поворота U
в текущем наборе не найдено. Различие относится к присутствию площадок и
их соединению, а не к повторному повороту координат.

<details><summary>Таблица 6. Площадки на руках, передаваемые текущей модели</summary>
<div class="audit-table"><table>
<thead><tr><th>Подключение ТТРКГ</th><th>Число площадок на двух руках</th><th>Число электрических групп</th></tr></thead>
<tbody>
<tr><td>1×1</td><td>4</td><td>4</td></tr>
<tr><td>1×4</td><td>10</td><td>4</td></tr>
<tr><td>2×2</td><td>8</td><td>4</td></tr>
<tr><td>Объединение уникальных координат этих вариантов</td><td>12</td><td>Зависит от подключения</td></tr>
</tbody></table></div></details>

Неподключённая металлическая площадка и отсутствие площадки на поверхности
задают разные физические условия. Для площадок без внешнего соединения
нужно явно определить модель: например, отдельный плавающий электрод
с нулевым суммарным внешним током, если это соответствует устройству контакта.
Этот выбор ещё не проверен по эксперименту. Величина влияния на боковой
импеданс при аудите не рассчитана.

**Исправление.** Хранить три самостоятельных описания: постоянные физические
площадки, электрические группы и состояние соединения для каждой записи.
Изменение подключения не должно молча удалять площадку. Для крестов также
нужен явный список физически оставшихся контактов; нынешний код CEM крестов
содержит восемь боковых площадок. Неизвестные состояния не следует заполнять
произвольным соединением.

Источники: [координаты на цилиндрах](../../MATLAB_TRKG4_real_subjects/tools/scorpion_georg_geometry_revision.py),
[подготовка вариантов монтажа](../../MATLAB_TRKG4_real_subjects/tools/georg_ct_fem_prepare.py),
[формирование CEM при боковом подборе](../../MATLAB_TRKG4_real_subjects/src/run_georg_lateral_inverse.m).

### 7. Стыки цилиндрических рук

**Геометрическая проверка пройдена в исследованных сечениях.** Радиус
цилиндров 58,887329 мм соответствует обхвату 370 мм. Аналитическая площадь
поперечного сечения составляет 10 894,156 мм², полигональная — 10 893,062 мм²;
различие около 0,010%. Astra проверила сечения на выбранной плоскости и
со стороны внутренней крышки: диск полностью покрыт исходным телом.
Недостаточная площадь соединения в этих проверках не обнаружена.

Для получения полного соединения центры цилиндров смещены в плоскости YZ
примерно на 18,71 и 14,65 мм, а плоскости заглублены на 23,5 и 16,5 мм.
Расстояния электродов при этом отсчитываются от прежней плоскости обрезки.
Токовые контакты находятся в 143,5 и 136,5 мм от новых заглублённых плоскостей,
хотя номинальное расстояние от прежней плоскости составляет 120 мм.

Это сохранённое геометрическое допущение, требующее ясного отображения обеих
плоскостей. Область вне поля зрения КТ назначается мягкотканным фоном,
включая искусственные руки. Их тканевой состав не восстановлен по КТ.
Следует отдельно оценить чувствительность к положению стыка и проводимости
искусственных рук. Номинальное расстояние нельзя незаметно переопределять
после заглубления.

Источник: [построение полного стыка цилиндров](../../MATLAB_TRKG4_real_subjects/tools/scorpion_georg_geometry_revision.py).

### 8. Маски тканей и проверка однородности

**В проверенных примерах ошибки сертификатора не обнаружены.** Эта проверка
выявляет тетраэдры, охватываемые воксельной областью с одной тканевой меткой,
и позволяет не повторять интегрирование для заведомо однородной области.
Astra сравнила результат с прямым перебором 1000 случайных небольших
воксельных коробок, включая частичный и полный выход за границы КТ;
несовпадений не найдено. Консервативное расширение коробки уменьшает число
сертифицированных элементов. Обобщённая безошибочность алгоритма этим
конечным тестом не доказана.

Формы и аффинные преобразования всех четырёх масок совпадают с КТ точно;
основной агент подтвердил это независимо. Пространственное совмещение
не подтверждает правильность анатомических границ. По проверке Astra
перекрытия есть только между жиром и другими масками: 804 вокселя с сердцем,
2825 с костями и 1003 с лёгкими. Воксели классифицируются с приоритетом
лёгких, затем костей, сердца и жира; незанятая область становится фоном.

В маске жира 11,2% вокселей имеют HU выше нуля. Это основание для локального
просмотра, но по этому одному признаку нельзя объявить сегментацию ошибочной:
нужно проверить границы, эффект частичного объёма и неоднородность тканей.
Нужна оценка толщины и HU-профиля жира непосредственно под каждой боковой
сборкой. Кости представлены одним материалом, а вся маска сердца — одним
эффективным сердечным материалом; отдельное разделение крови и миокарда
в данном боковом расчёте не выполнено.

Подготовка Ника использует аналогичный порядок масок и интегрирование долей.
Это подтверждает преемственность алгоритма, но не точность масок Георгия.
Индивидуальные операторы Ника на Георгия не переносятся.

Источники: [проверка однородности](../../MATLAB_TRKG4_real_subjects/tools/georg_lateral_quadrature.py),
[назначение тканевых меток](../../MATLAB_TRKG4_real_subjects/tools/scorpion_georg_fem_prepare.py),
[аналогичная подготовка жира у Ника](../../MATLAB_TRKG4_real_subjects/tools/lateral_fat_prepare.py).

### 9. Поле зрения КТ и локальная точность наружной поверхности

**Сохраняющиеся ограничения модели.** Лёгкие и сердце не достигают крайних
срезов по z, но жир и кости достигают обоих. Объём закрывается искусственными
поверхностями обрезки. Подготовленная геометрия охватывает исследованный
фрагмент тела с искусственными продолжениями рук.

Упрощение поверхности проверено 15 000 точек в каждом направлении.
Максимальные найденные расстояния между исходной и упрощённой поверхностями
составили 0,720 и 0,671 мм. Такой выборочный контроль не является доказанной
верхней границей ошибки всей поверхности. В визуальном паспорте отмечены
небольшие выступы неопределённого происхождения после восстановления контура.

Согласование объёмной сетки с выбранным STL хорошее: для исходной сетки
6 мм максимальное проверенное расстояние границы составляет 0,000112 мм,
объёмное расхождение — ноль. Проверка воспроизведения STL сеткой не заменяет
проверку самого STL по КТ. При диаметре контакта 5 мм локальная поверхность
под площадкой заслуживает отдельного контроля.

**Проверка.** Для каждого контакта показать одно КТ-сечение с HU, исходной
маской тела, конечным STL и тканевыми границами. Проверять также расстояние
монтажа до искусственных границ и их влияние отдельным численным сценарием.
Отдельное увеличение числа точек интегрирования не исправляет ошибочную
наружную геометрию.

Источники: [восстановление наружного контура](../../MATLAB_TRKG4_real_subjects/tools/georg_ct_model.py),
[проверка границы сетки](../../MATLAB_TRKG4_real_subjects/output/exploratory/georg_ct_model_20261011/mesh_h6_qc.local.json),
[визуальный паспорт контура](../../MATLAB_TRKG4_real_subjects/output/exploratory/georg_ct_model_20261011/contour_visual_review.local.json).

### 10. Единицы, материальные индексы и сохранённые операторы

**В текущем тракте ошибок преобразования не обнаружено.** Для назначения
тканей координаты LPS в миллиметрах переводятся в RAS и далее в воксельные
индексы; перед электрической сборкой миллиметры переводятся в метры.
Порядок строк оператора градиентов согласован с повторением каждой тканевой
доли три раза.

Порядок материалов сохранённых пятикомпонентных массивов — фон, лёгкие,
кости, сердце, жир. Свободные индексы 1, 2 и 5 соответствуют трём материалам
завершённого бокового поиска: фону, лёгким и жиру. В данном отчёте ρ₁, ρ₂
и ρ₃ обозначают именно эти три свободных сопротивления. Четырёхпараметрическая
постановка совместного анализа ТТРКГ и боковых сборок имеет другое соглашение
об обозначениях. Третий столбец старого материального массива соответствует
костям и не должен интерпретироваться как жир.

**Недостаток защиты будущего повторного использования.** Идентичность
материального кэша не включает явно настройку исходных единиц, настройку
масштабирования геометрии и контрольную сумму функции перевода в SI.
В текущем запуске они согласованы, но изменение единиц в будущем защищено
неполно. Кроме того, некоторые дочерние планы наследуют поля правил
интегрирования и области расчёта, не соответствующие их фактическим новым
операциям; исполняемые новые поля необходимо отличать от унаследованных.

**Исправление.** В паспорт каждого оператора включать исходные единицы,
фактический масштаб, версии используемых вспомогательных функций, число
точек интегрирования и контрольную сумму соответствующей таблицы долей.
Устаревшие поля уточнять в новом паспорте с сохранением исходного.

Источники: [материальная сборка](../../MATLAB_TRKG4_real_subjects/src/trkg4_georg_check_materials.m),
[перевод модели в SI](../../MATLAB_TRKG4_real_subjects/src/trkg4_scale_fmdl_to_si.m),
[контракт общей модели](../../MATLAB_TRKG4_real_subjects/tools/georg_model_contract.py).

### 11. Порядок исправлений и численных проверок

Приоритет задаётся тем, какую неопределённость должна устранить каждая
операция. Продолжение инверсии на ещё более крупной сетке до этого разделения
не даст ясного объяснения изменения импедансов.

<details><summary>Таблица 7. Рекомендуемые исправления и критерии следующего этапа</summary>
<div class="audit-table"><table>
<thead><tr><th>Порядок</th><th>Действие</th><th>Как проверить результат</th></tr></thead>
<tbody>
<tr><td>1</td><td>Исправить обработку пустых срезов и выбор родительского правила интегрирования; дополнить идентичность кэшей</td><td>Малые воспроизводимые примеры и обе пары правил; действующие сохранённые источники остаются неизменными</td></tr>
<tr><td>2</td><td>Разделить физические площадки, электрические группы и состояния подключения</td><td>Постоянный список контактов и отдельная схема каждой записи; численный эффект изменения граничных условий проверяется отдельно</td></tr>
<tr><td>3</td><td>Ввести количественный контроль формы элементов и локальный КТ-контроль под монтажами</td><td>Распределения качества, локальные HU-сечения и согласование контуров; новый кандидат сетки должен исправлять найденные плохие области</td></tr>
<tr><td>4</td><td>Разделить изменение пространства потенциала и переназначение тканевых долей на одной дочерней сетке</td><td>Родительская сетка; дочерняя с унаследованной проводимостью; та же дочерняя с новым отображением масок</td></tr>
<tr><td>5</td><td>Повторить пространственную проверку на пригодной сетке при фиксированных параметрах и контактах</td><td>Сохранённые допуски и оператор наблюдения; изменение критерия после провала не допускается</td></tr>
<tr><td>6</td><td>После выяснения источников ошибки продолжить совместный подбор</td><td>Новая версия модели и паспорт; прежние оценки не объявляются принятыми автоматически</td></tr>
</tbody></table></div></details>

Аудит установил проверяемые проблемы подготовки и подтвердил несколько
звеньев преобразования. Он не определил окончательную причину провала
сеточной сходимости и не валидировал индивидуальные тканевые сопротивления.
Общая геометрия, исходные координаты, исторические массивы и отрицательные
численные статусы сохранены.
